In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from numpy import array, nan, inf, float32

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
from dataclasses import dataclass

@dataclass
class EnsembleSpreadBounds:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]

@dataclass
class CompressionRatios:
    percentile: list[float]
    linquant: list[float]
    bitround: list[float]
    linquantquadstep: list[float]

In [4]:
def parse_single(path: Path):
    with Path(path).open() as f:
        for l in f.readlines():
            var, rest = l.split(": ", 1)
            info = eval(rest)

            yield var, info

In [5]:
def parse_pressure(path: Path):
    with Path(path).open() as f:
        ls = iter(f.readlines())
        
        l = None
        
        while True:
            if l is None:
                try:
                    l = next(ls)
                except:
                    break
        
            var, rest = l.split(":", 1)
        
            while True:
                try:
                    l = next(ls)
                except:
                    l = None
                    break
        
                if len(l.strip()) == 0:
                    continue
        
                if not l.startswith(" - "):
                    break
        
                level, rest = l[len(" - "):].split(": ", 1)
                info = eval(rest)
        
                yield var, level, info

In [6]:
single_ensemble_spread_bounds = dict(parse_single("ensemble-spread-single.txt"))

In [7]:
pressure_ensemble_spread_bounds = dict()
for var, level, info in parse_pressure("ensemble-spread-pressure.txt"):
    if var not in pressure_ensemble_spread_bounds:
        pressure_ensemble_spread_bounds[var] = dict()
    pressure_ensemble_spread_bounds[var][level] = info

In [8]:
import fcbench

In [9]:
fcbench.codecs.preload()

Loaded the fcbench.codecs.Asinh codec ...
Loaded the fcbench.codecs.BitRound codec ...
Loaded the fcbench.codecs.FixedOffsetScale codec ...
Loaded the fcbench.codecs.Identity codec ...
Loaded the fcbench.codecs.LinearQuantize codec ...
Loaded the fcbench.codecs.Log codec ...
Loaded the fcbench.codecs.RandomProjection codec ...
Loaded the fcbench.codecs.Reinterpret codec ...
Loaded the fcbench.codecs.Round codec ...
Loaded the fcbench.codecs.SwizzleReshape codec ...
Loaded the fcbench.codecs.Sz3 codec ...
Loaded the fcbench.codecs.UniformNoise codec ...
Loaded the fcbench.codecs.Zfp codec ...
Loaded the fcbench.codecs.Zlib codec ...
Loaded the fcbench.codecs.Zstd codec ...


In [10]:
lossless = fcbench.codecs.Zstd(level=3)

In [11]:
def compute_absolute_error_compression_ratio(mean: np.ndarray, eb_abs: float) -> float:
    quantized = np.round(mean / eb_abs).astype(int)
    compressed = lossless.encode(quantized - np.amin(quantized))

    return mean.nbytes / compressed.nbytes

def compute_relative_error_compression_ratio(mean: np.ndarray, eb_rel: float) -> float:
    info = np.finfo(mean.dtype)
    
    keepbits_mantissa = np.ceil(-np.log2(eb_rel * 2.0))
    keepbits_mantissa = int(max(0, min(keepbits_mantissa, info.bits - info.iexp - 1)))

    rounded = fcbench.codecs.BitRound(keepbits=keepbits_mantissa).encode(mean)
    compressed = lossless.encode(rounded)

    return mean.nbytes / compressed.nbytes

def compute_quadratic_error_compression_ratio(mean: np.ndarray, eb_qua: float, nbins=100) -> float:
    amin, amax = np.amin(mean), np.amax(mean)
    ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1)
    
    eb_abs = eb_qua - eb_qua * (-1.0 + 2.0 * (ibin+0.5)/(nbins))

    quantized = np.round(mean / eb_abs).astype(int)
    compressed = lossless.encode(quantized - np.amin(quantized))
    table = lossless.encode(ibin)

    return mean.nbytes / (compressed.nbytes + table.nbytes)
    

def compute_compression_ratios(mean: xr.DataArray, bounds: EnsembleSpreadBounds) -> CompressionRatios:
    mean = mean.copy(deep=True).values
    
    linquant = []
    for p, eb_abs in zip(bounds.percentile, bounds.absolute):
        if not np.isfinite(eb_abs):
            linquant.append(np.nan)
        else:
            linquant.append(compute_absolute_error_compression_ratio(mean, eb_abs))

    bitround = []
    for p, eb_rel in zip(bounds.percentile, bounds.relative):
        if not np.isfinite(eb_rel):
            bitround.append(np.nan)
        else:
            bitround.append(compute_relative_error_compression_ratio(mean, eb_rel))

    linquantquadstep = []
    for p, eb_qua in zip(bounds.percentile, bounds.quadratic):
        if not np.isfinite(eb_qua):
            linquantquadstep.append(np.nan)
        else:
            linquantquadstep.append(compute_quadratic_error_compression_ratio(mean, eb_qua))

    return CompressionRatios(
        percentile=bounds.percentile,
        linquant=linquant,
        bitround=bitround,
        linquantquadstep=linquantquadstep,
    )

In [12]:
from tqdm.notebook import tqdm

In [ ]:
with open("compression-ratio-single.txt", "w") as f:
    for key in tqdm(ds_single.mean, position=0):
        print(f"{key}: ", end="", flush=True, file=f)
        print(f"{compute_compression_ratios(ds_single.mean[key], single_ensemble_spread_bounds[key])!r}", flush=True, file=f)

  0%|          | 0/226 [00:00<?, ?it/s]

/tmp/junityre/27119206/ipykernel_1190184/2213447395.py:2: RuntimeWarning: invalid value encountered in cast
  quantized = np.round(mean / eb_abs).astype(int)
/tmp/junityre/27119206/ipykernel_1190184/2213447395.py:20: RuntimeWarning: invalid value encountered in cast
  ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1)
/tmp/junityre/27119206/ipykernel_1190184/2213447395.py:24: RuntimeWarning: invalid value encountered in cast
  quantized = np.round(mean / eb_abs).astype(int)


In [14]:
with open("compression-ratio-pressure.txt", "w") as f:
    for key in tqdm(ds_pressure.mean, position=0):
        print(f"{key}:", flush=True, file=f)
        for pressure in ds_pressure.mean[key].pressure_level.values:
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(f"{compute_compression_ratios(ds_pressure.mean[key].sel(pressure_level=pressure), pressure_ensemble_spread_bounds[key][f"{pressure} hPa"])!r}", flush=True, file=f)

  0%|          | 0/16 [00:00<?, ?it/s]

/tmp/junityre/27116134/ipykernel_1043266/2213447395.py:2: RuntimeWarning: invalid value encountered in divide
  quantized = np.round(mean / eb_abs).astype(int)
/tmp/junityre/27116134/ipykernel_1043266/2213447395.py:2: RuntimeWarning: invalid value encountered in cast
  quantized = np.round(mean / eb_abs).astype(int)
/tmp/junityre/27116134/ipykernel_1043266/2213447395.py:10: RuntimeWarning: divide by zero encountered in log2
  keepbits_mantissa = np.ceil(-np.log2(eb_rel * 2.0))
/tmp/junityre/27116134/ipykernel_1043266/2213447395.py:20: RuntimeWarning: invalid value encountered in divide
  ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1)
/tmp/junityre/27116134/ipykernel_1043266/2213447395.py:20: RuntimeWarning: invalid value encountered in cast
  ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1)
/tmp/junityre/27116134/ipykernel_1043266/2213447395.py:24: RuntimeWarning: invalid value encountered in divide
  quant